# 五步走完：Z3 解约束的标准流程，一步一步带你过

## 这一章带你走一遍流程

上一章已经看到，Z3 写约束满足问题，代码可以短得惊人。可短归短，套路得先摸清，这一章就按**基本流程**从第一步开始走。

- 用 Z3 解题的 5 个步骤
- Z3 的变量跟普通 Python 的数，差在哪儿
- 怎么从模型（也就是答案）里把值掏出来
- 例题：SEND + MORE = MONEY
- 把所有解都列出来的办法

这一章的代码请**从上往下依次执行**。前面代码里建的变量，后面的代码还要接着用。

In [1]:
from z3 import *

## 解题就是这五步

题目千变万化，用 Z3 解题的步骤却几乎不变。

| 步骤 | 要做的事 | 用到的工具 |
|---|---|---|
| 1 | 建变量 | `Int`, `Ints`, `IntVector` |
| 2 | 备一个求解器 | `Solver()` |
| 3 | 把约束加进去 | `solver.add()` |
| 4 | 问一声有没有解 | `solver.check()` |
| 5 | 把值取出来 | `solver.model()` |

这五步，我们一个一个看。

## 第 1 步：把变量立起来

先要有变量，装着那个还没定下来的值。造法有三条路。

In [2]:
x = Int("x")             # 创建一个整数变量
y, z = Ints("y z")       # 用空格分隔名字，一次创建多个
v = IntVector("v", 5)    # v[0] 〜 v[4] 共 5 个整数变量

造出来的变量，类型叫 `ArithRef`。`IntVector` 本身是 Python 的列表，只是里头装的一律是 `ArithRef`。

In [3]:
print(type(x), x, y, z)
print(type(v), type(v[0]), v)

<class 'z3.z3.ArithRef'> x y z
<class 'list'> <class 'z3.z3.ArithRef'> [v__0, v__1, v__2, v__3, v__4]


Z3 的变量跟 Python 里的数，脾气不一样。它的值还没定，所以你拿它去算，算不出数字来，式子原封不动地留着。

In [4]:
print(x + 1)            # 不会计算，原样显示为表达式
print(x + 1 == 10)      # 变成条件式（BoolRef）
print(type(x + 1 == 10))

x + 1
x + 1 == 10
<class 'z3.z3.BoolRef'>


## 第 2 步：备一个求解器

约束得有个地方攒着，还得有人替你去找解，这个角色就是 `Solver`。开局先建一个空的。

In [5]:
solver = Solver()
print(type(solver), solver)   # [] 表示「还没有任何约束」

<class 'z3.z3.Solver'> []


## 第 3 步：把约束一条条加进去

约束交给 `solver.add()`。`==`、`<`、`>`、`+` 这些运算符，写出来直接就是约束表达式。

一次塞一条也行，攒成列表一次性塞进去也行，效果一样。

In [6]:
solver = Solver()

constraints = [
    x + y == 10,   # x 与 y 之和是 10
    x > 0,         # x 是正数
    y > 0,         # y 是正数
]
solver.add(*constraints)   # 展开列表传入（等同于 solver.add(a, b, c)）
print(solver)

[x + y == 10, x > 0, y > 0]


把 `solver` 交给 `print()`，它会把眼下装着的约束全列出来。想核对条件有没有写岔，看一眼就明白。

## 第 4 步：问一声有没有解

`Solver.check()` 问的就是这句话：满足这些约束的解，到底存不存在。回答一共三种。

| 返回值 | 含义 |
|---|---|
| `sat` | 找到解了（satisfiable） |
| `unsat` | 这样的解根本不存在（unsatisfiable） |
| `unknown` | 判断不出来（超时了、掺进了 Z3 不擅长的式子，等等） |

In [7]:
if solver.check() == sat:
    print("有解")
else:
    print("无解")

有解


## 第 5 步：从模型里把值掏出来

有解的时候，`solver.model()` 能给你具体的数值。它交回来的是一个对象，叫 `ModelRef`。

In [8]:
if solver.check() == sat:
    model = solver.model()
    print("x =", model[x])
    print("y =", model[y])
else:
    print("无解")

x = 9
y = 1


`model[x]` 拿到手的是 Z3 自己的 `IntNumRef` 类型，直接当 Python 整数去加减是不行的。想拿它算数，得先过一道 `as_long()` 转换。

In [9]:
print(type(model), type(model[x]))

print("x =", model[x].as_long())
print("x + y =", model[x].as_long() + model[y].as_long())

<class 'z3.z3.ModelRef'> <class 'z3.z3.IntNumRef'>
x = 9
x + y = 10


(z3-send-more-money)=
## 例题：SEND + MORE = MONEY

[用 OrTools 解同一个问题](ortools-send-more-money)

这是一类把字母换成数字的谜题，叫字母算术（Cryptarithm）。

> S, E, N, D, M, O, R, Y 各自代表 0〜9 里**不同的**数字。
>
> $$SEND + MORE = MONEY$$
>
> 请找出让这个式子成立的数字组合。

答案给的是 $9567 + 1085 = 10652$。Z3 真能算到这儿吗？我们上手试。

解法不用另找，就是这一章那五步。

1. 把 S, E, N, D, M, O, R, Y 立成整数变量
2. 加上约束：每个字母都在 0〜9 之间，而且彼此不同
3. 把单词翻成算式：$SEND = 1000S + 100E + 10N + D$
4. 把 $SEND + MORE = MONEY$ 当约束加进去
5. 用 `check()` 和 `model()` 把解取出来

In [10]:
# 把 S, E, N, D, M, O, R, Y 设为整数变量
S, E, N, D, M, O, R, Y = Ints("S E N D M O R Y")
letters = [S, E, N, D, M, O, R, Y]

solver = Solver()

# 1. 每个字母都是 0〜9 之一
for letter in letters:
    solver.add(0 <= letter, letter <= 9)

# 2. 字母不同，数字也不同
solver.add(Distinct(letters))

# 3. 把单词写成算式  例：SEND = 1000*S + 100*E + 10*N + D
send = 1000*S + 100*E + 10*N + D
more = 1000*M + 100*O + 10*R + E
money = 10000*M + 1000*O + 100*N + 10*E + Y

# 4. SEND + MORE = MONEY
solver.add(send + more == money)

# 5. 最高位不能是 0
solver.add(S != 0, M != 0)

if solver.check() == sat:
    model = solver.model()
    print({str(letter): model[letter].as_long() for letter in letters})
    print(f"{model.eval(send)} + {model.eval(more)} = {model.eval(money)}")
else:
    print("无解")

{'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O': 0, 'R': 8, 'Y': 2}
9567 + 1085 = 10652


答案只有一种。整段代码里，运算符只用了两个。

| 运算符 | 作用 | 例子 |
|---|---|---|
| `And` | 让所有条件同时成立 | `And(x > 0, y > 0)` |
| `Distinct` | 让给定的一批变量取值彼此不同 | `Distinct(a, b, c)` |

取值也有两条路。

- `model[x]` … 取变量此刻的值
- `model.eval(式)` … 取整个算式算出来的结果

### 模型里装着什么

`ModelRef` 就是把变量和值拢在一处的容器。往 `[]` 里递什么，回给你的东西就不一样。

| 传入 | 返回 |
|---|---|
| 变量（`ArithRef`） | 它的值（`IntNumRef`） |
| 下标（整数） | 变量的声明（`FuncDeclRef`） |

In [11]:
print(model[0], type(model[0]))   # 第 0 个变量的「声明」

decl = model[0]
print(decl(), type(decl()))       # 加上 () 就变成变量（ArithRef）

S <class 'z3.z3.FuncDeclRef'>
S <class 'z3.z3.ArithRef'>


`FuncDeclRef` 是变量的声明，后面补一对 `()`，才变回变量本身。至于哪个字母排在第 0 位，取决于 Z3 内部的顺序，说不定每次跑都不一样。

摸清了这个门道，模型里所有变量的值就能一把全取出来。

In [12]:
{str(decl): model[decl].as_long() for decl in model}

{'S': 9, 'O': 0, 'N': 6, 'R': 8, 'E': 5, 'M': 1, 'Y': 2, 'D': 7}

## 把所有的解都列出来

要的不是“一个解”，而是“全部解”，办法是：每找到一个，就补一条**“这个解不许再出现”**的约束，然后接着 `check()`。

1. `check()` 问一声有没有解
2. 有解就用 `model()` 取出来
3. 补一条“刚才那个解别再给了”，回到第 1 步
4. 等到没解了（`unsat`），收工

In [13]:
abc = Ints("a b c")
solver = Solver()

for n in abc:
    solver.add(1 <= n, n <= 3)   # 都在 1〜3 之间
solver.add(Distinct(abc))        # 三个数互不相同

while solver.check() == sat:
    model = solver.model()
    print(model)
    # 禁止与刚找到的解「完全相同」的组合
    solver.add(Not(And([decl() == model[decl] for decl in model])))

[c = 2, b = 3, a = 1]
[a = 3, c = 1, b = 2]
[c = 2, b = 1, a = 3]
[c = 1, b = 3, a = 2]
[c = 3, a = 2, b = 1]
[a = 1, c = 3, b = 2]


`Not(And(c1, c2, ...))` 跟 `Or(Not(c1), Not(c2), ...)` 是同一句话的两种写法。落到约束上，意思就是“随便哪一项不同都行”，也就是“别再给我刚才那个解”。

下面这一行，效果一模一样。

```python
solver.add(Or([decl() != model[decl] for decl in model]))
```

只不过这么写，每找出一个解就要叫一次 `check()`。解一多，时间就压不住了，这点得留心。

## 收个尾

用 Z3 解题，永远是这五步。

1. 建变量（`Int`, `Ints`, `IntVector`）
2. 建 `Solver()`
3. 用 `solver.add()` 把约束加进去
4. 用 `solver.check()` 看是 `sat` / `unsat` / `unknown`
5. 用 `solver.model()` 取值，再过 `as_long()` 变成 Python 整数

- Z3 的变量装的是“还没定下来的值”，所以它一直以式的样子存在
- 用 `print(solver)` 把约束摊开看一遍，写错的地方往往一眼就能发现
- 想要全部解，就补上禁止已找到解的约束，一轮一轮再来